# Frame the Problem and Meet the Dataset

Notebook for **Chapter 1** of the *ways* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ways/ch01.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
# Setup: the files used in this chapter are downloaded next to the notebook.
import os, urllib.request
os.makedirs("data", exist_ok=True)
for f in ["customers.csv", "orders.csv", "products.csv", "shopkart.db"]:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve("https://dadipavan.github.io/Learning/study/ways/data/" + f, "data/" + f)


## 1.1 The path from raw data to decisions

Think of cooking for a dinner party. You decide the menu (framing), buy and store ingredients (data engineering), wash and trim them (cleaning), taste and combine (analysis), check a recipe change with a fair taste test (statistics), cook tomorrow's menu from patterns of what guests ate (machine learning), try an exotic technique if it adds something (deep learning, GenAI), and finally serve, and watch whether guests keep coming back (operations and monitoring). Nobody starts with the exotic technique. Nobody serves unwashed vegetables.

## 1.2 From an email to ten measurable questions

"Orders are up but I do not feel richer" is not a question a computer can answer. A measurable question names a **metric** (a number computed in a defined way), a **slice** (period, city, segment) and a **comparison** (against what). The process has four steps.

## 1.3 The dataset: dictionary and structure

ShopKart has three tables. The data was generated by a script (`tools/ways/make_data.py`, a fixed random seed) so that every reader gets identical files, but it contains the faults of real data on purpose: spelling variants, missing values, duplicate rows, impossible values and extreme outliers. Nothing in it is real customer data.

## 1.4 Loading the data in every tool

Loading looks trivial and is where many silent errors begin: a date read as text, an id read as a number with a decimal point, a leading zero lost, an encoding wrong. For each tool we show the exact action, then compare the first answer: how many rows?

In [ ]:
DATA = "data/"
import csv, sqlite3
import pandas as pd

# 1. pandas
df = pd.read_csv(DATA + "orders.csv", parse_dates=["order_date"])
print("pandas :", df.shape, df.order_date.min().date(), df.order_date.max().date())

# 2. plain Python
with open(DATA + "orders.csv", newline="") as f:
    rows = list(csv.DictReader(f))
dates = [r["order_date"] for r in rows]
print("python :", (len(rows), len(rows[0])), min(dates), max(dates))

# 3. SQL (SQLite)
con = sqlite3.connect(DATA + "shopkart.db")
print("sql    :", con.execute("SELECT COUNT(*), MIN(order_date), MAX(order_date) FROM orders").fetchone())

pandas : (20150, 14) 2024-01-01 2024-12-31
python : (20150, 14) 2024-01-01 2024-12-31
sql    : (20150, '2024-01-01', '2024-12-31')


## 1.5 A first profile in two tools

A profile is a one-page health check: how many rows, how many missing values per column, how many distinct values in key columns, what ranges numbers have. It costs minutes and prevents hours of confusion. The full cleaning is Chapter 3; here we only look.

In [ ]:
DATA = "data/"
import pandas as pd
df = pd.read_csv(DATA + "orders.csv", parse_dates=["order_date"])
print("rows", len(df), "| distinct order_id", df.order_id.nunique(), "| distinct customers", df.customer_id.nunique())
print("fully duplicated rows:", df.duplicated().sum())
print("missing per column:")
print(df.isna().sum()[lambda s: s > 0].to_string())
print("distinct city spellings:", df.city.nunique())
print(df[["quantity", "unit_price", "delivery_minutes"]].describe().loc[["min", "50%", "max"]].round(1).to_string())

rows 20150 | distinct order_id 20000 | distinct customers 3216
fully duplicated rows: 150
missing per column:
city                 120
channel               61
unit_price            90
promised_minutes    1998
delivery_minutes    2791
distinct city spellings: 18
     quantity  unit_price  delivery_minutes
min       1.0        40.0             -23.0
50%       2.0       330.0              39.0
max      99.0      3425.0            2900.0


In [ ]:
DATA = "data/"
import sqlite3
con = sqlite3.connect(DATA + "shopkart.db")
q = """
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT order_id) AS n_order_ids,
       COUNT(DISTINCT customer_id) AS n_customers,
       SUM(city IS NULL) AS city_missing,
       SUM(unit_price IS NULL) AS price_missing,
       SUM(channel IS NULL) AS channel_missing,
       COUNT(DISTINCT city) AS city_spellings,
       MIN(quantity) AS qmin, MAX(quantity) AS qmax,
       MIN(delivery_minutes) AS dmin, MAX(delivery_minutes) AS dmax
FROM orders"""
cur = con.execute(q)
for name, val in zip([d[0] for d in cur.description], cur.fetchone()):
    print(f"{name:18s}{val}")

n_rows            20150
n_order_ids       20000
n_customers       3216
city_missing      120
price_missing     90
channel_missing   61
city_spellings    18
qmin              1
qmax              99
dmin              -23.0
dmax              2900.0


## 1.6 The expected answers: raw versus cleaned

Before analysis, write down what each tool *should* return. You will check this table again in Chapters 3 and 4, in four tools. Notice how far the naive answer is from the cleaned one: this is the reason the next two chapters exist.

In [ ]:
DATA = "data/"
import sys
sys.path.insert(0, DATA)
import prep
raw, cust, prods = prep.load_raw(DATA)
o, cust, prods = prep.load_clean(DATA)
naive_rev = (raw.quantity * raw.unit_price * (1 - raw.discount)).sum()
print(f"raw rows            {len(raw):>12,}")
print(f"clean orders        {len(o):>12,}")
print(f"naive revenue       {naive_rev:>12,.0f}   (all rows, no cleaning, missing prices skipped)")
print(f"clean net revenue   {o.revenue.sum():>12,.0f}   (delivered only)")
print(f"avg order value     {o[o.status=='delivered'].net.mean():>12,.1f}")
print(f"return rate         {(o.status=='returned').mean():>12.4f}")
print(f"late rate           {o.late.mean():>12.4f}   ({o.late.notna().sum():,} orders with both times)")
print(f"repeat customers    {(o.groupby('customer_id').size() >= 2).mean():>12.4f}")

raw rows                  20,150
clean orders              19,995
naive revenue         22,404,947   (all rows, no cleaning, missing prices skipped)
clean net revenue     17,349,916   (delivered only)
avg order value          1,073.2
return rate               0.1476
late rate                 0.1376   (17,208 orders with both times)
repeat customers          0.8096


## 1.7 Ethics, privacy and scope

Every stage touches people. A customer id looks harmless until it is joined with age, city and order times; then it can single out an individual. Four habits protect people and you.

## 1.8 Choosing tools: a first map

The same Q1 to Q8 can be answered by any tool in the table. The tool changes who can do it, how fast, and how repeatable it is.

## Chapter recap

## Practice set

Problem 1.9.1 Name the eight stages of the path and the chapter where each is covered.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ways/ch01.html